# **Adult Income Prediction using Scikit-learn Pipelines**
**Objective**

Build a complete machine learning pipeline to predict whether a person's annual income exceeds $50K using the Adult Census Income dataset.

In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [5]:
df = pd.read_csv('adult.csv')

In [6]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [8]:
(df == '?').sum()
# These are the Missing Values stored like '?'

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64

In [9]:
df = df.replace('?',np.nan)

In [10]:
df.isnull().sum()

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64

In [11]:
df.info()
# Now pandas is able to indentify the missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        46043 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       46033 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   47985 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


# **Train-Test Split**

In [12]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns= ['income']),df['income'],test_size = 0.2,random_state = 42)

# **Data Preprocessing**


## Imputation Transformer
Missing values in categorical columns are replaced using the most frequent value.

In [13]:
trf1 = ColumnTransformer([
    ('impute_workclass',SimpleImputer(strategy = 'most_frequent'),[1]),
    ('impute_occupation',SimpleImputer(strategy = 'most_frequent'),[5]),
    ('impute_nativecountry',SimpleImputer(strategy = 'most_frequent'),[13])
],remainder = 'passthrough')

## One-Hot Encoding
Convert categorical variables into numerical features suitable for machine learning algorithms.

In [14]:
trf2 = ColumnTransformer([
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
     [0, 1, 2, 5, 7, 8, 9, 10])
], remainder='passthrough')

## Feature selection
Select the most informative features using the Chi-Square statistical test.

In [15]:
trf3 = SelectKBest(score_func=chi2,k=8)

## Train The Model
Decision Trees are simple, interpretable, and work well with mixed numerical and categorical features.

In [16]:
trf4 = DecisionTreeClassifier()

# Create Pipeline

In [17]:
pipe = Pipeline([
    ('trf1' , trf1),
    ('trf2' , trf2),
    ('trf3' , trf3),
    ('trf4' , trf4),
])

In [ ]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_workclass',
                                  SimpleImputer(strategy='most_frequent'), [1]),
                                 ('impute_occupation',
                                  SimpleImputer(strategy='most_frequent'), [5]),
                                 ('impute_nativecountry',
                                  SimpleImputer(strategy='most_frequent'),
                                  [13])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [0, 1, 2, 5, 7, 8, 9, 10])]),
 'trf3': SelectKBest(k=8, score_func=<function chi2 at 0x7ec5b2d06de0>),
 'trf4': DecisionTreeClassifier()}

In [ ]:
pipe.fit(X_train,y_train)
print("Model trained successfully.")

Model trained successfully.


In [20]:
y_pred = pipe.predict(X_test)

In [21]:
y_pred

array(['<=50K', '<=50K', '>50K', ..., '<=50K', '<=50K', '>50K'],
      shape=(9769,), dtype=object)

In [22]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.8172791483263384

# **Conclusion**
**Successfully built an end-to-end Scikit-learn Pipeline that performs:**

(1)Missing value imputation

(2)One-hot encoding

(3)Feature selection using Chi-Square

(4)Decision Tree classification

The pipeline ensures all preprocessing steps are automatically applied during both training and prediction.